# Classifiers Playground
"How can I make a classification model that can classify four seconds of 8-channel EEG data into either movement or no movement?"

The suspiciously LDA shaped horse: 🙂‍↕️🙂‍↕️🙂‍↕️

In [1]:
print("test")
import numpy as np
from mne.decoding import CSP
import mne
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

test


In [4]:
# config
DATA_FOLDER = "../data/"
DATA_SESSION = "3-4/joshfoot/"
SESSIONS = [7, 8, 9]
CHANNELS = [2, 3, 4, 6] # in theory these are the only ones that should matter

In [5]:
# load filtered data
alltrials = np.empty((0, 2))
for session in SESSIONS:
    filtered_session = np.load(f"{DATA_FOLDER}{DATA_SESSION}filtered-session-{session}.npy", allow_pickle=True)
    for i in range(len(filtered_session)):
        filtered_session[i][1] = filtered_session[i][1][CHANNELS, 62:-62]
    alltrials = np.concatenate((alltrials, filtered_session), axis=0)

In [4]:
# set up csp and lda labels
labels = np.array([1 if trial[0] == 'clench right foot' else 0 for trial in alltrials])
eeg = np.array([trial[1] for trial in alltrials])

print(labels.shape)
print(eeg.shape)

(60,)
(60, 4, 1001)


In [5]:
# set up line that has a pipe
csp = CSP(n_components=4, reg='ledoit_wolf', log=True)
lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')

pl = Pipeline([
    ('csp', csp),
    ('lda', lda)
])

In [6]:
# yall remember pa3 with the folding nonsense
# turns out there's straight up a built in library for that

# the highly intelligent people who invented mne made logs super verbose by default
mne.set_log_level('ERROR')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pl, eeg, labels, cv=cv, scoring='accuracy')
print(f"fold scores: {scores}")

fold scores: [0.5        0.83333333 0.83333333 0.58333333 0.66666667]


In [7]:
X_train, X_test, y_train, y_test = train_test_split(eeg, labels, test_size=0.20, random_state=42)
pl.fit(X_train, y_train)

train_score = pl.score(X_train, y_train)
print(f"train score: {train_score}")
test_score = pl.score(X_test, y_test)
print(f"test score: {test_score}")


train score: 0.75
test score: 0.75
